In [1]:
from einops import rearrange
from tqdm import tqdm
import numpy as np
import torch
import numpy as np
import torch
import os
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
import sklearn
from sklearn.metrics import accuracy_score,\
    classification_report, confusion_matrix, roc_auc_score, f1_score, precision_score, precision_recall_curve, recall_score, auc
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import json
from torch.nn.functional import one_hot
from torch.utils.data import DataLoader, TensorDataset
import time
from sklearn.neural_network import MLPClassifier

In [18]:
def get_head_data(path_head, path_label):
    heads = np.load(path_head)
    head_wise_activations = heads.reshape(heads.shape[0], 32, 32, -1)
    labels = np.load(path_label, allow_pickle=True)

    return head_wise_activations, labels

In [22]:
model_name = 'llama2-7b-chat-hf'
path_head = f'../data/feature/gsm8k_{model_name}_heads.npy'
path_label = f'../data/feature/gsm8k_{model_name}_labels.npy'

head_activations, labels = get_head_data(path_head, path_label)

head_wise_activations_train = head_activations[:800, :, :, :]
labels_train = labels[:800]
head_wise_activations_test = head_activations[200:, :, :, :]
labels_test = labels[200:]

In [24]:
labels_test

array([1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1,
       1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, None,
       1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1,
       1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1,
       1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1,
       1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1,
       0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, None, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0,
       0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0,

In [23]:
''' 
MLP
'''
layer_num = 32
head_num = 32
m_auc = np.empty([layer_num,head_num], dtype = float) 
m_auroc = np.empty([layer_num,head_num], dtype = float) 

d_clf = {}
for layer in tqdm(range(layer_num)):
    for head in range(head_num):
        X_train = head_wise_activations_train[:, layer, head, :]
        Y_train = labels_train
        assert X_train.shape[0]==Y_train.shape[0]

        clf = LogisticRegression(max_iter=500).fit(X_train, Y_train) 

        X_test = head_wise_activations_test[:, layer, head, :]
        Y_test = labels_test
        assert X_test.shape[0]==Y_test.shape[0]

        if layer not in d_clf:
            d_clf[layer] = {}
        if head not in d_clf[layer]:
            d_clf[layer][head] = clf

        tempPredicts = clf.predict(X_test)
        tempLogits = [i[1] for i in clf.predict_proba(X_test)]

        precision_list, recall_list, _ = precision_recall_curve(Y_test, tempLogits)
        auroc = roc_auc_score(Y_test, tempLogits)
        prauc = auc(recall_list, precision_list)

        m_auc[layer][head] = prauc
        m_auroc[layer][head] = auroc

  0%|          | 0/32 [00:00<?, ?it/s]


ValueError: Unknown label type: unknown. Maybe you are trying to fit a classifier, which expects discrete classes on a regression target with continuous values.